# Experiment 006 — Path-First Native CLM

**Gate 0 only; NOT REGISTERED. Default mode does not train or read a corpus.**
Import this notebook into the existing `bingjiangli/minicells` editor. Use Internet
for the pinned public source checkout; preflight runs on CPU. No new Kaggle note,
Actions API launch, setup deadline or application training cap is used.

After reviewed registration and explicit development approval, the same notebook
can run one arm or two isolated T4 workers. Decode profiles run serially after both
workers exit. Existing `GITHUB_TOKEN` supplies the established write preflight and
Askpass publication path; `HF_TOKEN` is optional for separately approved storage.
Prepare/mount the frozen **006** asset bundle first; no historical asset is reused.
Confirmation stays sealed. Training, sparsity and path diagrams are not scientific support.


In [ ]:
from pathlib import Path
import json
import os
import re
import subprocess
import sys
import threading

MODE = 'preflight'  # preflight | dry-run | train | train_pair | profile | publish_existing | analyze
SOURCE_COMMIT = '__PIN_SOURCE_COMMIT__'
CONFIG = 'research/08-hybrid-clm/experiments/006-path-first-clm/config/proposed.json'
SEED = None  # only a reviewed, registered development/confirmation seed
ROLE = 'development'
ARM = 'R2'
ARM_PAIR = ('R0', 'R1')
ROOT = Path('/kaggle/working/minicells-exp006')
ASSET_ROOT = Path('/kaggle/input/minicells-006-registered-assets')
RUN_ROOT = Path('/kaggle/working/minicells-exp006-runs')
REPOSITORY_URL = 'https://github.com/ArcheLabs/mini-cells.git'
KAGGLE_NOTEBOOK_URL = 'https://www.kaggle.com/code/bingjiangli/minicells'

if MODE not in {'preflight','dry-run','train','train_pair','profile','publish_existing','analyze'}:
    raise ValueError('Unsupported mode')
if not re.fullmatch('[0-9a-f]{40}', SOURCE_COMMIT):
    raise ValueError('Import the released Notebook with its full pinned source SHA')

def git(*args):
    # Per-command network timeout; never a deadline carried across training.
    return subprocess.run(['git',*args],cwd=ROOT,check=True,text=True,
                          stdout=subprocess.PIPE,timeout=300).stdout.strip()

if not ROOT.exists():
    ROOT.mkdir(parents=True)
    git('init')
    git('remote','add','origin',REPOSITORY_URL)
    git('fetch','--depth=1','origin',SOURCE_COMMIT)
    git('checkout','--detach',SOURCE_COMMIT)
else:
    if not (ROOT/'.git').is_dir():
        raise RuntimeError('Existing folder has no Git checkout; preserve it and choose a fresh ROOT')
    git('merge-base','--is-ancestor',SOURCE_COMMIT,'HEAD')

sys.path[:0] = [str(ROOT),str(ROOT/'research/08-hybrid-clm/experiments/006-path-first-clm/implementation')]
from exp006.config import load, authorize
c,p,_ = load(ROOT/CONFIG)
print({'experiment':'006-path-first-clm','source':SOURCE_COMMIT,'status':p['status'],'mode':MODE,'scientific_decision':False})

# This gate intentionally occurs before secrets, asset reads or GPU access.
if MODE in {'train','train_pair','profile','publish_existing'}:
    authorize(c,p,SEED,ROLE)
    if ARM not in p['arms'] or any(a not in p['arms'] for a in ARM_PAIR):
        raise ValueError('Arm is not in registered protocol')
    if MODE=='train_pair' and (len(ARM_PAIR)!=2 or len(set(ARM_PAIR))!=2):
        raise ValueError('Select two distinct registered arms')

    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    if not os.environ.get('GITHUB_TOKEN','').strip():
        try:
            os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN').strip()
        except Exception as exc:
            raise RuntimeError('Existing GITHUB_TOKEN is unavailable in this editor') from exc
    if not os.environ['GITHUB_TOKEN'].strip():
        raise RuntimeError('GITHUB_TOKEN is empty')
    if not os.environ.get('HF_TOKEN','').strip():
        try:
            os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN').strip()
        except Exception:
            pass  # local frozen assets do not require HF access

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'


In [ ]:
def command(mode,arm=None,output=None):
    cmd=[sys.executable,str(ROOT/'scripts/research/run_experiment_006.py'),
         '--mode',mode,'--config',str(ROOT/CONFIG)]
    if arm is not None:
        cmd += ['--arm',arm,'--seed',str(SEED),'--role',ROLE,
                '--source-commit',SOURCE_COMMIT,'--assets',str(ASSET_ROOT),
                '--output',str(output),'--device','cuda:0']
    return cmd

def publish(output=None,preflight=False):
    cmd=[sys.executable,str(ROOT/'scripts/research/publish_experiment_006.py'),
         '--config',str(ROOT/CONFIG),'--seed',str(SEED),'--role',ROLE,
         '--source-commit',SOURCE_COMMIT]
    cmd += ['--preflight-only'] if preflight else ['--output',str(output)]
    subprocess.run(cmd,cwd=ROOT,check=True)

def stream_worker(cmd,physical_gpu,label):
    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(physical_gpu),PYTHONUNBUFFERED='1')
    process=subprocess.Popen(cmd,cwd=ROOT,env=env,text=True,
                             stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
    def pump():
        for line in process.stdout:
            print('['+label+'] '+line.rstrip(),flush=True)
    thread=threading.Thread(target=pump);thread.start()
    return process,thread

def output_for(arm):
    return RUN_ROOT/SOURCE_COMMIT/('seed-'+str(SEED))/ROLE/arm

if MODE in {'preflight','dry-run'}:
    subprocess.run(command(MODE),cwd=ROOT,check=True)
elif MODE=='analyze':
    report=RUN_ROOT/('analysis-'+SOURCE_COMMIT+'-'+str(SEED)+'.json')
    cmd=[sys.executable,str(ROOT/'scripts/research/analyze_experiment_006.py'),
         '--config',str(ROOT/CONFIG),'--source-commit',SOURCE_COMMIT,'--output',str(report)]
    if SEED is not None:cmd += ['--seed',str(SEED)]
    subprocess.run(cmd,cwd=ROOT,check=True)
elif MODE=='publish_existing':
    publish(output_for(ARM))  # no training/profile repeat required
else:
    publish(preflight=True)  # established Git write check before GPU work
    arms=ARM_PAIR if MODE=='train_pair' else (ARM,)
    if MODE in {'train','train_pair'}:
        import torch
        if torch.cuda.device_count()<len(arms):
            raise RuntimeError('Select GPU T4 x2 for a pair, or use single-arm mode')
        workers=[]
        for gpu,arm in enumerate(arms):
            out=output_for(arm)
            if (out/'metrics.json').exists():
                print(arm+': completed training found; skip retraining',flush=True)
                continue
            workers.append((arm,*stream_worker(command('train',arm,out),gpu,arm)))
        failures=[]
        for arm,process,thread in workers:
            result=process.wait();thread.join()
            if result:failures.append(arm)
        # All workers have exited; publish recoverable incomplete evidence first.
        for arm in failures:
            publish(output_for(arm))
        if failures:raise RuntimeError('Incomplete arm(s): '+', '.join(failures)+'; checkpoints preserved')
    # Serial isolated profiling after every training worker has exited.
    for arm in arms:
        out=output_for(arm)
        if not (out/'decode-profile.json').exists():
            process,thread=stream_worker(command('profile',arm,out),0,arm+' profile')
            result=process.wait();thread.join()
            if result:
                publish(out)
                raise RuntimeError('Profile incomplete; retry profile without retraining')
        publish(out)
print({'notebook_phase':'completed','mode':MODE,'scientific_decision':False})


## Status and recovery

Preflight output `ENGINEERING_READY` means only that engineering fixtures passed.
The protocol remains `PROPOSED_NOT_REGISTERED` until a separate approval revision.
Do not select training merely because preflight is green.

Keep the Kaggle working outputs (`latest.pt`, metadata and learning curves). Model
weights and corpus are not pushed to Git. Publication uses source-specific
`results/006-path-first-clm-<source prefix>` branches. `publish_existing` retries
archived evidence without training; failed/incomplete evidence is immutable and
separate from completed arms. No CPU/GPU timing is reused as a scientific result.
